In [9]:
#Make imports
import numpy as np
import re
import pickle
import os
import seaborn as sns
import string

In [10]:
#TPU settings
%tensorflow_version 2.x
import tensorflow as tf
print("Tensorflow version " + tf.__version__)

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.
Tensorflow version 2.17.0


In [11]:
def preprocess(text):
  text = ''.join(ch for ch in text if ch not in string.punctuation)
  text = text.lower()
  text = re.sub(r'\d','',text)
  text = re.sub(r'\s+',' ',text)
  text = text.strip()
  return text

In [12]:
#Extract dataset and preprocess
dataset_root = "./drive/MyDrive/Parallel/"

if os.path.exists(dataset_root + "EngAmh.pickle"):
  with open(dataset_root + "EngAmh.pickle", 'rb') as f:
    english_sentences, amharic_sentences = pickle.load(f)
else:
  if not os.path.exists(dataset_root + "EngAmhE.txt"):
    os.system(dataset_root)

  with open(dataset_root + "EngAmhE.txt",'r') as f:
    english_sentences = f.read().split('\n')

  with open(dataset_root + "EngAmhA.txt",'r') as f:
    amharic_sentences = f.read().split('\n')

  english_sentences = [preprocess(en) for en in english_sentences]
  amharic_sentences = ['<START> ' + re.sub('[a-zA-Z]','',preprocess(am)) + ' <END>' for am in amharic_sentences]

  #Remove duplicate sentences
  english_unique = set()
  english_sentences_temp = []
  amharic_sentences_temp = []
  l = len(english_sentences)
  for i in range(l):
    if english_sentences[i] not in english_unique:
      english_unique.add(english_sentences[i])
      english_sentences_temp.append(english_sentences[i])
      amharic_sentences_temp.append(amharic_sentences[i])

  english_sentences = english_sentences_temp
  amharic_sentences = amharic_sentences_temp

  with open(dataset_root + "EngAhm.pickle",'wb') as f:
    pickle.dump((english_sentences, amharic_sentences), f)

In [13]:
print(len(english_sentences), len(amharic_sentences))
print()
english_sentences[:5], amharic_sentences[:5]

5356 5356



(['the commercial bank of ethiopia cbe is a testament to ethiopias capacity in building institutions and passing them on to future generations',
  'it is delighting to see that the bank on its th anniversary completed this stateoftheart building',
  'there is the need to build institutions that continue to learn from the past and redeem seasons',
  'in order for our banks to compete with international banks they need far more modernization in terms of manpower and operation aside erecting such magnificent buildings',
  'our stabilization efforts'],
 ['<START> የኢትዮጵያ ንግድ ባንክ ኢትዮጵያ ተቋማትን መገንባትና ለትውልድ ማሸገጋር እንደምትችል ማሳያ ሆኖ ቆይቷል። <END>',
  '<START> በኛ ዓመት የምስረታ ክብረ በዓሉ ለቀልጣፋ አሠራር የተመቸ ሕንጻን አጠናቅቆ ለዚህ ማብቃቱ ደስ ያሰኛል። <END>',
  '<START> የትናንትን ሳይረሱ ትምሕርት መቅሰማቸውን የሚቀጥሉና ዘመንን የሚዋጁ ተቋማትን መገንባት ያስፈልጋል። <END>',
  '<START> ባንኮቻችን ከዓለም አቀፍ ባንኮች ጋር ለመፎካከር እንዲችሉ ከሕንጻ ግንባታ ባለፈ በሰው ሀይልና በአሰራር ሊዘምኑ ያስፈልጋል። <END>',
  '<START> ሰላም የማስፈንና የማረጋጋት ጥረታችን <END>'])

In [14]:
vocab_size = 10000
total_sentences = 5500
maxlen = 40
epochs = 50
validation_split = 0.10

In [15]:
en_data = []
am_data = []

cnt = 0

for (en,am) in zip(english_sentences, amharic_sentences):
  l = min(len(en.split()), len(am.split()))
  if l <= maxlen:
    en_data.append(en)
    am_data.append(am)
    cnt += 1
  if cnt == total_sentences:
    break

In [16]:
#Tokenize the texts and convert to sequences
en_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='', oov_token='<OOV>', lower=False)
en_tokenizer.fit_on_texts(en_data)
en_sequences = en_tokenizer.texts_to_sequences(en_data)

am_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='', oov_token='<OOV>', lower=False)
am_tokenizer.fit_on_texts(am_data)
am_sequences = am_tokenizer.texts_to_sequences(am_data)

english_vocab_size = len(en_tokenizer.word_index) + 1
amharic_vocab_size = len(am_tokenizer.word_index) + 1
print("English Vocab Size: ", english_vocab_size)
print("Amharic Vocab Size: ", amharic_vocab_size)

English Vocab Size:  12637
Amharic Vocab Size:  28496


In [17]:
#Prepare encoder data
encoder_inputs = tf.keras.preprocessing.sequence.pad_sequences(en_sequences, maxlen=maxlen, padding='post')

In [18]:
#Prepare decoder data
decoder_inputs = []
decoder_outputs = []

for am in am_sequences:
  decoder_inputs.append(am[:-1])
  decoder_outputs.append(am[1:])

decoder_inputs = tf.keras.preprocessing.sequence.pad_sequences(decoder_inputs, maxlen=maxlen, padding='post')
decoder_outputs = tf.keras.preprocessing.sequence.pad_sequences(decoder_outputs, maxlen=maxlen, padding='post')

In [19]:
# Training and Testing split
# 90%, 10%
split = int(0.90 * total_sentences)

X_train = [encoder_inputs[:split], decoder_inputs[:split]]
y_train = decoder_outputs[:split]

# Test data to evaluate our NMT model using BLEU score
X_test = en_data[:split]
y_test = am_data[:split]

print(X_train[0].shape, X_train[1].shape, y_train.shape)

(4950, 40) (4950, 40) (4950, 40)


In [20]:
#Define LSTM model
d_model = 256

#Encoder
inputs = tf.keras.layers.Input(shape=(None,))
x = tf.keras.layers.Embedding(english_vocab_size, d_model, mask_zero=True)(inputs)
_,state_h,state_c = tf.keras.layers.LSTM(d_model,activation='relu',return_state=True)(x)

#Decoder
targets = tf.keras.layers.Input(shape=(None,))
embedding_layer = tf.keras.layers.Embedding(amharic_vocab_size, d_model, mask_zero=True)
x = embedding_layer(targets)
decoder_lstm = tf.keras.layers.LSTM(d_model,activation='relu',return_sequences=True, return_state=True)
x,_,_ = decoder_lstm(x, initial_state=[state_h, state_c])
dense1 = tf.keras.layers.Dense(amharic_vocab_size, activation='softmax')
x = dense1(x)

model = tf.keras.models.Model(inputs=[inputs, targets],outputs=x)
model.summary()

loss = tf.keras.losses.SparseCategoricalCrossentropy()
model.compile(optimizer='rmsprop', loss=loss, metrics=['accuracy'])

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, None)           │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_layer_1             │ (None, None)           │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding (Embedding)     │ (None, None, 256)      │      3,235,072 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ not_equal (NotEqual)      │ (None, None)           │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_1 (Embedding)   │ (None, None, 256)      │      7,294,976 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm (LSTM)               │ [(None, 256), (None,   │        525,312 │ embedding[0][0],       │
│                           │ 256), (None, 256)]     │                │ not_equal[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_1 (LSTM)             │ [(None, None, 256),    │        525,312 │ embedding_1[0][0],     │
│                           │ (None, 256), (None,    │                │ lstm[0][1], lstm[0][2] │
│                           │ 256)]                  │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, None, 28496)    │      7,323,472 │ lstm_1[0][0]           │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 18,904,144 (72.11 MB)

 Trainable params: 18,904,144 (72.11 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
!pip install pyyaml h5py

In [22]:
#Save model after each epoch
save_model_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='./drive/MyDrive/Parallel/en-am.keras', # Added .keras extension to the filepath
    monitor='val_accuracy',
    mode='max'
)

In [23]:
model.fit(X_train, y_train, epochs=epochs, validation_split=validation_split, callbacks=[save_model_callback, tf.keras.callbacks.TerminateOnNaN()])

Epoch 1/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 490s 3s/step - accuracy: 0.0235 - loss: 10.0376 - val_accuracy: 0.0248 - val_loss: 9.1050
Epoch 2/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 495s 3s/step - accuracy: 0.0256 - loss: 8.8300 - val_accuracy: 0.0257 - val_loss: 8.9797
Epoch 3/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 501s 3s/step - accuracy: 0.0281 - loss: 8.6008 - val_accuracy: 0.0240 - val_loss: 8.9572
Epoch 4/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 507s 3s/step - accuracy: 0.0280 - loss: 8.4548 - val_accuracy: 0.0261 - val_loss: 8.9210
Epoch 5/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 506s 3s/step - accuracy: 0.0293 - loss: 8.3006 - val_accuracy: 0.0265 - val_loss: 9.0032
Epoch 6/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 498s 3s/step - accuracy: 0.0294 - loss: 8.1677 - val_accuracy: 0.0267 - val_loss: 8.9379
Epoch 7/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 499s 3s/step - accuracy: 0.0298 - loss: 8.0687 - val_accuracy: 0.0215 - val_loss: 9.1178
Epoch 8/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 504s 3s/step - accuracy: 0.0300 - loss: 7.9214 - val_acc

In [27]:
#Retrieve previously saved stuff
saved_model = tf.keras.models.load_model('./drive/MyDrive/Parallel/en-am.keras')

saved_model.summary()

inputs = saved_model.get_layer('input_layer').output  # Changed from 'input_1' to 'input_layer'
_,state_h,state_c = saved_model.get_layer('lstm').output
targets = saved_model.get_layer('input_layer_1').output # Changed from 'input_2' to 'input_layer_1'
embedding_layer = saved_model.get_layer('embedding_1')
decoder_lstm = saved_model.get_layer('lstm_1')
dense1 = saved_model.get_layer('dense')

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, None)           │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_layer_1             │ (None, None)           │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding (Embedding)     │ (None, None, 256)      │      3,235,072 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ not_equal (NotEqual)      │ (None, None)           │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_1 (Embedding)   │ (None, None, 256)      │      7,294,976 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm (LSTM)               │ [(None, 256), (None,   │        525,312 │ embedding[0][0],       │
│                           │ 256), (None, 256)]     │                │ not_equal[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_1 (LSTM)             │ [(None, None, 256),    │        525,312 │ embedding_1[0][0],     │
│                           │ (None, 256), (None,    │                │ lstm[0][1], lstm[0][2] │
│                           │ 256)]                  │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, None, 28496)    │      7,323,472 │ lstm_1[0][0]           │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 37,808,290 (144.23 MB)

 Trainable params: 18,904,144 (72.11 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 18,904,146 (72.11 MB)

In [28]:
#Inference Model

#Encoder
encoder = tf.keras.models.Model(inputs, [state_h, state_c])

#Decoder
decoder_input_h = tf.keras.layers.Input(shape=(d_model,))
decoder_input_c = tf.keras.layers.Input(shape=(d_model,))
x = embedding_layer(targets)
x, decoder_output_h, decoder_output_c = decoder_lstm(x, initial_state=[decoder_input_h, decoder_input_c])
x = dense1(x)
decoder = tf.keras.models.Model([targets] + [decoder_input_h, decoder_input_c],
                                [x] + [decoder_output_h, decoder_output_c])

In [44]:
def predict_sentence(en_input):
  input_seq = en_tokenizer.texts_to_sequences([en_input])

  # Convert input_seq to a NumPy array
  input_seq = np.array(input_seq)

  next_h, next_c = encoder.predict(input_seq)

  curr_token = np.zeros((1,1)) # Changed from np.zeros(1)
  curr_token[0,0] = am_tokenizer.word_index['<START>'] # Changed from curr_token[0]

  pred_sentence = ''

  for i in range(maxlen):
    output, next_h, next_c = decoder.predict([curr_token] + [next_h, next_c])
    next_token = np.argmax(output[0, 0, :])
    next_word = am_tokenizer.index_word[next_token]
    if next_word == '<END>':
      break
    else:
      pred_sentence += ' ' + next_word
      curr_token[0,0] = next_token # Changed from curr_token[0]

  return pred_sentence

In [45]:
#Testing and Analysis
import nltk

candidates = []
references = []

ctr = 20
i = 0

while ctr>0:
  l = len(X_test[i].split())
  if l<=maxlen:   #Choose only sentences of length in range [5,15]
    pred_sentence = predict_sentence(X_test[i])
    candidates.append(pred_sentence.split())

    print("Input: ", X_test[i])
    print("Prediction: ", pred_sentence)

    #google_translated_sentence = translate_client.translate(X_test[i], target_language='hi')['translatedText']

    #print("Google Translated Reference: ", google_translated_sentence)
    print("Dataset Reference: ", ' '.join(y_test[i].split()[1:-1]))
    print()
    references.append([y_test[i].split()[1:-1]])

    ctr -= 1
  i += 1

print(nltk.translate.bleu_score.corpus_bleu(references, candidates))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Input:  the commercial bank of ethiopia cbe is a testament to ethiopias capacity in building institutions and passing them on to future generations
Prediction:   ኢትዮጵያ የኢትዮ የኢትዮጵያ ብሔራዊ ዳይሬክተር ታዋቂ ጸሐፊ ማሞ ወልዴ ቴክኖሎጂ ክፍሎች አባላት ለመጀመር ዝግጅቱን ዝግጅቱን ተጠቁሟል።
Dataset Reference:  የኢትዮጵያ ንግድ ባንክ ኢትዮጵያ ተቋማትን መገንባትና ለትውልድ ማሸገጋር እ

/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_